In [2]:
import os
import json
from tqdm import tqdm

def read_json_file(path: str) -> dict | list:
    with open(path, 'r') as f:
        return json.load(f)

def write_json_file(data: dict | list, path: str) -> None:
    with open(path, 'w') as f:
        json.dump(data, f, indent=4, ensure_ascii=False)

In [ ]:
raw_data = read_json_file("./data/raw_data/Generation/alpaca_dataset.json")
for d in raw_data:
    d["instruction"] = d["instruction"].replace("<Generate_Response>", "").replace("</Generate_Response>", "").strip()
    d["instruction"] = json.loads(d["instruction"])
    d["output"] = json.loads(d["output"])

print(len(raw_data))
raw_data[0]

1746


{'instruction': {'title': 'راهنمای جامع اپلیکیشن بانکت: خدمات نوین بانکی و سرویس\u200cهای ارزش افزوده',
  'language': 'PERSIAN',
  'context': 'IMPORTANT DOCUMENTS:\n[Doc 0]\nQuestion: `تحویل فیزیکی طلا`, Answer: بدلیل شرایط جنگی کشور، تحویل فیزیکی طلا تا اطلاع ثانوی غیرفعال شده است.\n\n\nREGULAR DOCUMENTS:\n[Doc 0]\nراهنمای خرید و فروش طلا در اپلیکیشن بانکت\nتحویل فیزیکی طلا\nعیار طلای تحویل شده و انتخاب آن\n**عیار طلای تحویل شده:**\n\n\n\nاگرچه خرید و فروش طلا در اپلیکیشن بانکت، فقط براساس قیمت طلا با عیار 18 انجام میشود،\nاما هنگام تحویل فیزیکی، امکان انتخاب انواع شمش\u200cها با عیار 18 و 24 وجود دارد.\nدرصورت انتخاب شمش با عیار 24، میزان موجودی طلای کاربر،\nاز عیار 18 به عیار 24 تبدیل شده و متناسب با آن، امکان انتخاب شمش، وجود خواهد داشت.\n\t Reference:\n\t 7_Gold_V_5.docx\n\n[Doc 1]\nراهنمای خرید و فروش طلا در اپلیکیشن بانکت\nتسویه وجه حاصل از فروش طلا\nفرآیند تسویه وجه پس از فروش طلا\n## تسویه وجه طلا – تسویه وجه حاصل از فروش طلا:\n\n\n\nپس از فروش طلا و ارسال درخواست برداشت وجه ا

In [4]:
def format_input(input_data: dict) -> str:
    text = f"""<GenerationResponse>\
<name>{input_data['name']}</name>\
<title>{input_data['title']}</title>\
<context>{input_data['context']}</context>\
<history>{input_data['history']}</history>\
<message>{input_data['message']}</message>\
<language>{input_data['language']}</language>\
</GenerationResponse>"""    
    return text

In [5]:
def convert_to_alpaca_format(raw_data: list) -> list:
    final_data = []
    for item in tqdm(raw_data):
        final_data.append({
            "instruction": format_input(item["instruction"]),
            "input": "",
            "output": json.dumps(item["output"], ensure_ascii=False)
        })
    return final_data

In [6]:
alpaca_data = convert_to_alpaca_format(raw_data)
write_json_file(alpaca_data, "./data/processed_data/GenerationTag_Alpaca.json")

alpaca_data[0]

In [8]:
data_tags = {}

for d in alpaca_data:
    data_tags.setdefault(json.loads(d["output"])["tag"], []).append(d)

for k, v in data_tags.items():
    print(f"{k}: {len(v)}")


small_data = []
small_data += data_tags["not_enough"]
small_data += data_tags["normal"][:250]

print(len(small_data))
small_data[0]

In [9]:
write_json_file(small_data, "./data/processed_data/GenerationTag_Alpaca374.json")